<a href="https://colab.research.google.com/github/Priyanshiix12/MLE/blob/main/Hospital_Readmission_Prediction_Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Hospital Readmission Prediction
### Logistic Regression with L2 Regularization

This notebook uses the Kaggle **Readmission Dataset** to predict whether a patient will be readmitted within 30 days.

We will:
- Explore the dataset
- Prepare numerical and categorical features
- Train Logistic Regression with L2 regularization
- Evaluate using ROC-AUC
- Analyze different classification thresholds
- Compare precision and recall
- Discuss false positives vs false negatives in a clinical context


In [1]:
# ============================================================
# 1️⃣ INSTALL KAGGLEHUB
# ============================================================

!pip -q install kagglehub


In [2]:
# ============================================================
# 2️⃣ DOWNLOAD DATASET FROM KAGGLE
# ============================================================

import kagglehub

path = kagglehub.dataset_download(
    "vanpatangan/readmission-dataset"
)

print("Dataset downloaded to:")
print(path)


100%|██████████| 47.6k/47.6k [00:00<00:00, 16.2MB/s]

Extracting files...
Dataset downloaded to:
/root/.cache/kagglehub/datasets/vanpatangan/readmission-dataset/versions/1


In [3]:
# ============================================================
# 3️⃣ IMPORT LIBRARIES
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

import matplotlib.pyplot as plt


In [4]:
# ============================================================
# 4️⃣ CHECK DOWNLOADED FILES
# ============================================================

print("Files in dataset folder:")

for file in os.listdir(path):
    print(file)


Files in dataset folder:
test_df.csv
sample_submission.csv
train_df.csv


In [5]:
# ============================================================
# 5️⃣ LOAD TRAINING DATA
# ============================================================

train_path = os.path.join(path, "train.csv")

df = pd.read_csv(train_path)

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())


FileNotFoundError: [Errno 2] No such file or directory: '/root/.cache/kagglehub/datasets/vanpatangan/readmission-dataset/versions/1/train.csv'

In [ ]:
# ============================================================
# 6️⃣ UNDERSTAND THE DATASET
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nReadmission values:")
print(df["readmitted"].value_counts())


In [ ]:
# ============================================================
# 7️⃣ CHECK MISSING VALUES
# ============================================================

missing_values = df.isnull().sum()

print("Missing values:")
display(
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
)


In [ ]:
# ============================================================
# 8️⃣ CREATE BINARY TARGET
# ============================================================

# <30 means readmitted within 30 days.
# 1 → Readmitted within 30 days
# 0 → Not readmitted within 30 days

df["readmitted_30_days"] = (
    df["readmitted"] == "<30"
).astype(int)

print("New target distribution:")
print(df["readmitted_30_days"].value_counts())

print("\nNew target percentage:")
print(
    df["readmitted_30_days"]
    .value_counts(normalize=True) * 100
)


In [ ]:
# ============================================================
# 9️⃣ SEPARATE FEATURES AND TARGET
# ============================================================

# X = patient information used for prediction
# y = answer we want to predict

X = df.drop(
    ["readmitted", "readmitted_30_days"],
    axis=1
)

y = df["readmitted_30_days"]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# ============================================================
# 🔟 REMOVE IDENTIFIER COLUMNS
# ============================================================

# IDs identify a patient/encounter rather than describing
# useful patient characteristics.

id_columns = [
    "encounter_id",
    "patient_nbr"
]

X = X.drop(
    columns=[
        col for col in id_columns
        if col in X.columns
    ]
)

print("Features after removing IDs:")
print(X.columns.tolist())


In [ ]:
# ============================================================
# 1️⃣1️⃣ IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Number of categorical features:",
      len(categorical_features))

print("Number of numerical features:",
      len(numerical_features))

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)


In [ ]:
# ============================================================
# 1️⃣2️⃣ CREATE PREPROCESSING PIPELINES
# ============================================================

# Numerical data:
# Missing values → median
# Then standardize the values

numerical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Categorical data:
# Missing values → most frequent value
# Then convert categories into numbers

categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)


In [ ]:
# ============================================================
# 1️⃣3️⃣ COMBINE PREPROCESSING USING COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_preprocessor,
            numerical_features
        ),
        (
            "cat",
            categorical_preprocessor,
            categorical_features
        )
    ]
)


In [ ]:
# ============================================================
# 1️⃣4️⃣ CREATE MACHINE LEARNING PIPELINE
# ============================================================

# Raw patient data
#       ↓
# Preprocessing
#       ↓
# Logistic Regression

model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                max_iter=1000
            )
        )
    ]
)

# L2 regularization is the default for LogisticRegression.
# C=1.0 controls the inverse strength of regularization.


In [ ]:
# ============================================================
# 1️⃣5️⃣ TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:
# ============================================================
# 1️⃣6️⃣ TRAIN THE MODEL
# ============================================================

model.fit(
    X_train,
    y_train
)

print("Model training completed!")


In [ ]:
# ============================================================
# 1️⃣7️⃣ PREDICT READMISSION PROBABILITIES
# ============================================================

# predict_proba gives probabilities for both classes.
# [:, 1] selects probability of class 1:
# readmitted within 30 days.

y_probability = model.predict_proba(
    X_test
)[:, 1]

print("Minimum probability:", y_probability.min())
print("Maximum probability:", y_probability.max())
print("Mean probability:", y_probability.mean())


In [ ]:
# ============================================================
# 1️⃣8️⃣ CALCULATE ROC-AUC
# ============================================================

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("ROC-AUC:", roc_auc)


In [ ]:
# ============================================================
# 1️⃣9️⃣ SET CLASSIFICATION THRESHOLD
# ============================================================

threshold = 0.30

y_pred = (
    y_probability >= threshold
).astype(int)

print("Classification threshold:", threshold)


In [ ]:
# ============================================================
# 2️⃣0️⃣ CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


In [ ]:
# ============================================================
# 2️⃣1️⃣ CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)


In [ ]:
# ============================================================
# 2️⃣2️⃣ CONFUSION MATRIX VISUALIZATION
# ============================================================

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Not Readmitted",
        "Readmitted"
    ]
)

disp.plot()

plt.title("Hospital Readmission Confusion Matrix")
plt.show()


In [ ]:
# ============================================================
# 2️⃣3️⃣ EXTRACT TN, FP, FN, TP
# ============================================================

tn, fp, fn, tp = cm.ravel()

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)


In [ ]:
# ============================================================
# 2️⃣4️⃣ ROC CURVE
# ============================================================

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_probability
)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Hospital Readmission")
plt.legend()
plt.show()


In [ ]:
# ============================================================
# 2️⃣5️⃣ ANALYZE PERFORMANCE AT DIFFERENT THRESHOLDS
# ============================================================

thresholds_to_test = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

results = []

for threshold in thresholds_to_test:

    # Convert probabilities into 0/1 predictions
    predictions = (
        y_probability >= threshold
    ).astype(int)

    # Calculate confusion matrix values
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    # Calculate recall
    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    # Calculate precision
    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    # Store results
    results.append({
        "Threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Recall": recall,
        "Precision": precision
    })

threshold_df = pd.DataFrame(results)

threshold_df["Recall"] = threshold_df["Recall"].round(3)
threshold_df["Precision"] = threshold_df["Precision"].round(3)

display(threshold_df)


## 🏥 Clinical Interpretation

**False Negative:** The model predicts that a patient is not high risk, but the patient is actually readmitted within 30 days. In this case, the model may miss a patient who could benefit from additional follow-up.

**False Positive:** The model predicts that a patient is high risk, but the patient is not actually readmitted. This can lead to additional monitoring or unnecessary use of healthcare resources.

**Threshold trade-off:** Lowering the threshold generally increases recall and helps catch more actual readmissions, but it can also increase false positives. The final threshold should be selected using clinical priorities, the relative cost of false negatives and false positives, and available hospital resources.
